<a href="https://colab.research.google.com/github/mwaikelvin2k-creator/Financial_risk_analysis/blob/main/financial_loan_risk_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Loan Risk — Analysis Summary
**FinTech Innovations | Loan Approval Prediction**

---

## Overview

This project builds a machine learning model to automate loan approval decisions for FinTech Innovations. Three classifiers were trained and tuned — Decision Tree, Random Forest, and AdaBoost. The final model is a **tuned AdaBoost Classifier** achieving **99% precision and AUC of 1.00**, with only **1 false positive** on the test set. Given that approving a bad loan costs \$50,000 vs \$8,000 for denying a good one, minimising false positives was the primary business goal.

---

## 1. Business Understanding

**Problem:** FinTech Innovations currently uses a manual loan approval process that is slow, inconsistent, and prone to costly errors.

**Two types of errors and their costs:**

| Error | Description | Cost |
|-------|-------------|------|
| False Positive | Approved a bad loan (defaulted) | \$50,000 |
| False Negative | Denied a good loan (lost profit) | \$8,000 |

**Conclusion:** False positives are 6x more costly. We optimise for **Precision** as the primary metric.

**Task type:** Binary Classification (LoanApproved: 0 = Denied, 1 = Approved)

## 2. Data Understanding

- **Dataset size:** 20,000 rows
- **Features:** Age, AnnualIncome, CreditScore, Experience, LoanAmount, LoanDuration, NumberOfDependents, MonthlyDebtPayments, CreditCardUtilizationRate, NumberOfOpenCreditLines, NumberOfCreditInquiries, DebtToIncomeRatio, PreviousLoanDefaults, PaymentHistory, LengthOfCreditHistory, and categorical features.
- **Target:** `LoanApproved` (0 or 1)

**Data cleaning:**
- `AnnualIncome` had dollar signs and commas — stripped and converted to float.

**Key EDA findings:**

- **Target distribution:** Slight class imbalance — more denied (0) than approved (1) loans.
- **CreditScore vs Loan Approval:** Approved applicants have a higher median credit score (~595 vs ~575). There is overlap between groups, meaning credit score alone does not determine approval.
- **Strong correlations found:**
  - Age & Experience (~0.96) — multicollinearity risk
  - LoanAmount & LoanDuration — expected relationship
  - CreditScore & DebtToIncomeRatio (negative) — higher debt lowers credit score
  - CreditScore & PreviousLoanDefaults (negative) — more defaults = worse credit

## 3. Data Preparation

A `ColumnTransformer` pipeline was built with two separate preprocessing flows:

**Numerical features:**
- `StandardScaler` — normalises all numeric features to the same scale
- `SimpleImputer(strategy='median')` — fills missing values with median to avoid skew

**Categorical features:**
- `OneHotEncoder(drop='first')` — converts string categories to numeric (drops first to avoid dummy variable trap)
- `SimpleImputer(strategy='most_frequent')` — fills missing values with mode (appropriate for categorical/ordinal data)

**Train/Test Split:** 80% train, 20% test, stratified on target, `random_state=42`

## 4. Modelling Results

Three models were trained as base pipelines, checked for overfitting with cross-validation, then tuned with `GridSearchCV` (cv=5, scoring='precision').

### Model Comparison

| Model | False Positives (Base) | False Positives (Tuned) | Precision | AUC |
|-------|----------------------|------------------------|-----------|-----|
| Decision Tree | 23 | 19 | 1.00 | 1.00 |
| Random Forest | 13 | 5 | 0.99 | 0.99 |
| **AdaBoost** | **5** | **1** | **0.99** | **1.00** |

**Cross-validation scores (no overfitting detected):**
- Decision Tree: Mean ~0.97, low variance
- Random Forest: Mean ~0.98, low variance  
- AdaBoost: Mean ~0.99, low variance

### Hyperparameters Tuned

| Model | Parameters Tuned |
|-------|------------------|
| Decision Tree | max_depth, min_samples_split, min_samples_leaf, criterion, class_weight |
| Random Forest | n_estimators, max_depth, min_samples_split, min_samples_leaf, class_weight |
| AdaBoost | n_estimators, learning_rate |

## 5. Final Model — Tuned AdaBoost Classifier

**Selected because:** Fewest false positives (1) and highest AUC (1.00) after tuning — directly aligned with the business goal of minimising costly bad loan approvals.

**Final test set performance:**

| Metric | Class 0 (Denied) | Class 1 (Approved) |
|--------|-----------------|--------------------|
| Precision | 0.99 | 0.99 |
| Recall | 0.99 | 0.99 |
| F1-Score | 0.99 | 0.99 |
| AUC | — | 1.00 |

**Confusion Matrix (Final Model):**
```
              Predicted 0    Predicted 1
Actual 0         3043             1        ← Only 1 bad loan approved
Actual 1            6           950
```

**Business impact of the final model:**
- 1 false positive × \$50,000 = **\$50,000 in bad loan exposure**
- 6 false negatives × \$8,000 = **\$48,000 in missed profit**
- Total error cost ≈ **\$98,000** — far less than a manual process at scale

## 6. Conclusion & Recommendations

**The tuned AdaBoost model is recommended for deployment.**

**Key takeaways:**
1. CreditScore, DebtToIncomeRatio, and PreviousLoanDefaults are the most predictive features based on EDA.
2. AdaBoost outperformed Decision Tree and Random Forest in the metric that matters most — precision (minimising false positives).
3. Cross-validation confirmed stable performance — no overfitting.

**Potential improvements:**
- Investigate Age/Experience multicollinearity — dropping one may simplify the model without loss of performance.
- Test XGBoost or Gradient Boosting as additional ensemble candidates.
- Monitor for data drift over time as loan applicant profiles change.
- Consider a cost-sensitive threshold tuning approach to directly optimise for the \$50k vs \$8k cost ratio.